# 🎓 Re:Learn — Misconception Classifier Training in Google Colab
### Fine-Tuning DistilBERT with Paired Hard Negatives on Cloud GPU

- **Research Backing**: Brown & Altadmri (2017) 100M+ events; Perkins (1986) Bugs vs Slips
- **Zero Setup**: Dataset is generated directly in-memory — zero file uploads required!


In [ ]:
# Step 1: Install Hugging Face & ML libraries
%pip install -q transformers datasets accelerate scikit-learn matplotlib pandas


In [ ]:
# Step 2: Verify GPU Acceleration
import torch
print('PyTorch Version:', torch.__version__)
if torch.cuda.is_available():
    print(f'✅ GPU Accelerated Runtime Active: {torch.cuda.get_device_name(0)}')
else:
    print('ℹ️ Running on CPU runtime (fast for our lightweight classifier).')


In [ ]:
# Step 3: Pure Python Dataset Generator (100% In-Memory, Never Hangs!)
import pandas as pd

seed_data = [
    # M-01: Assignment vs Comparison
    ("x = 5\nif x = 5:\n    print(x)", "SyntaxError: invalid syntax", "(none)", "M-01"),
    ("count = 10\nif count = 10:\n    print('ten')", "SyntaxError: invalid syntax", "(none)", "M-01"),
    ("val = 3\nif val = 3:\n    pass", "SyntaxError: invalid syntax", "(none)", "M-01"),
    ("score = 100\nif score = 100:\n    print('win')", "SyntaxError: invalid syntax", "(none)", "M-01"),
    ("target = 5\nif target = 5:\n    print('hit')", "SyntaxError: invalid syntax", "(none)", "M-01"),
    ("while x = 1:\n    print(x)", "SyntaxError: invalid syntax", "(none)", "M-01"),

    # M-09: PAIRED HARD NEGATIVES (Identical error string, but motor slip)
    ("score = 10\nif score == 10:\n    print('win')\ntarget = 5\nif target = 5:\n    print('match')", "SyntaxError: invalid syntax", "(none)", "M-09"),
    ("a = 1\nif a == 1:\n    pass\nb = 2\nif b = 2:\n    print(b)", "SyntaxError: invalid syntax", "(none)", "M-09"),
    ("if x == 5 and y == 10:\n    pass\nif z = 20:\n    print(z)", "SyntaxError: invalid syntax", "(none)", "M-09"),
    ("pritn('hello')", "NameError: name 'pritn' is not defined", "(none)", "M-09"),
    ("prnt('result')", "NameError: name 'prnt' is not defined", "(none)", "M-09"),
    ("total = 20\nlabel = 'Total: ' + str(total)\nitems = 3\nsummary = 'Items: ' + items", "TypeError: can only concatenate str (not 'int') to str", "(none)", "M-09"),
    ("items = [1, 2, 3]\nprint(items[0])\nprint(items[3])", "IndexError: list index out of range", "(none)", "M-09"),
    ("for i in range(3):\n    print(i)\nwhile n < 5\n    n += 1", "SyntaxError: expected ':'", "(none)", "M-09"),

    # M-02: Off-by-one / Range Bounds
    ("for i in range(1, 10):\n    print(i)", "(none)", "1..9 (expected 1..10)", "M-02"),
    ("nums = [10, 20, 30]\nfor i in range(1, 4):\n    print(nums[i])", "IndexError: list index out of range", "(none)", "M-02"),
    ("for i in range(5):\n    print(i)", "(none)", "0..4 (expected 1..5)", "M-02"),
    ("arr = [1, 2, 3]\nprint(arr[3])", "IndexError: list index out of range", "(none)", "M-02"),

    # M-03: Mutable List Aliasing
    ("a = [1, 2]\nb = a\nb.append(3)\nprint(a)", "(none)", "[1, 2, 3] (expected [1, 2])", "M-03"),
    ("original = [10, 20]\nbackup = original\nbackup.append(30)\nprint(original)", "(none)", "[10, 20, 30] (expected [10, 20])", "M-03"),
    ("def add(item, bag=[]):\n    bag.append(item)\n    return bag\nadd(1); print(add(2))", "(none)", "[1, 2] (expected [2])", "M-03"),
    ("x = [1]\ny = x\ny[0] = 99\nprint(x[0])", "(none)", "99 (expected 1)", "M-03"),

    # M-04: And/Or Return Values
    ("x = 5 or 3\nprint(x)", "(none)", "5 (expected True)", "M-04"),
    ("y = 0 or 'fallback'\nprint(y)", "(none)", "fallback (expected False)", "M-04"),
    ("flag = 'hello' and 'world'\nprint(flag)", "(none)", "world (expected True)", "M-04"),

    # M-05: Type Confusion (Str + Num)
    ("age = '5' + 3\nprint(age)", "TypeError: can only concatenate str (not 'int') to str", "(none)", "M-05"),
    ("price = 20\nmsg = 'Total is: ' + price", "TypeError: can only concatenate str (not 'int') to str", "(none)", "M-05"),
    ("total = '10' + 5", "TypeError: can only concatenate str (not 'int') to str", "(none)", "M-05"),

    # M-06: Loop Invariant / Never Updates
    ("n = 0\nwhile n < 5:\n    print(n)", "TimeoutError: infinite loop detected", "(none)", "M-06"),
    ("i = 1\nwhile i <= 10:\n    print('running')", "TimeoutError: infinite loop detected", "(none)", "M-06"),
    ("n = 0\nwhile n < 5\n    print(n)", "SyntaxError: expected ':'", "(none)", "M-06"),

    # M-07: Indentation Scope
    ("for i in range(3):\nprint(i)", "IndentationError: expected an indented block", "(none)", "M-07"),
    ("def greet(name):\nprint(name)", "IndentationError: expected an indented block", "(none)", "M-07"),
    ("if True:\nprint('ok')", "IndentationError: expected an indented block", "(none)", "M-07"),

    # M-08: Print vs Return
    ("def double(x):\n    print(x * 2)\ny = double(5)\nprint(y + 1)", "TypeError: unsupported operand type(s) for +: 'NoneType' and 'int'", "(none)", "M-08"),
    ("def calc(a, b):\n    print(a + b)\nres = calc(2, 3) * 10", "TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'", "(none)", "M-08"),
    ("def get_val():\n    print(42)\nx = get_val()\nif x > 10:\n    print('big')", "TypeError: '>' not supported between instances of 'NoneType' and 'int'", "(none)", "M-08")
]

# Expand systematically across variable names
expanded = []
for code, err, out, label in seed_data:
    expanded.append({'student_code': code, 'error_message': err, 'wrong_output': out, 'misconception_id': label})
    for v1, v2 in [('count', 'total'), ('x', 'y'), ('score', 'points'), ('val', 'num'), ('items', 'data'), ('age', 'price'), ('target', 'limit')]:
        sub_c = code.replace('target', v1).replace('nums', v2).replace('a', v1).replace('x', v2)
        expanded.append({'student_code': sub_c, 'error_message': err, 'wrong_output': out, 'misconception_id': label})

df = pd.DataFrame(expanded).drop_duplicates(subset=['student_code', 'misconception_id'])
df.to_csv('misconceptions_expanded.csv', index=False)
print(f'✅ Successfully generated {len(df)} verified student samples in memory!')
print('Taxonomy breakdown:')
print(df['misconception_id'].value_counts())


In [ ]:
# Step 4: Multi-Modal Context Representation & 75/25 Split
from sklearn.model_selection import train_test_split
from datasets import Dataset  # type: ignore # pyright: ignore[reportMissingImports]

df['text'] = (
    df['student_code'].astype(str) + 
    ' [ERR] ' + df['error_message'].fillna('(none)').astype(str) + 
    ' [OUT] ' + df['wrong_output'].fillna('(none)').astype(str)
)

labels = sorted(df['misconception_id'].unique())
lab2id = {l: i for i, l in enumerate(labels)}
id2lab = {i: l for l, i in lab2id.items()}
df['label'] = df['misconception_id'].map(lab2id)

train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=42, stratify=df['label']
)
print(f'Training samples: {len(train_df)} | Held-Out Test samples: {len(test_df)}')


In [ ]:
# Step 5: Tokenize with DistilBERT
from transformers import AutoTokenizer  # type: ignore # pyright: ignore[reportMissingImports]

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=256
    )

train_ds = Dataset.from_pandas(train_df).map(tokenize_batch, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize_batch, batched=True)


In [ ]:
# Step 6: Fine-Tune DistilBERT Misconception Classifier
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer  # type: ignore # pyright: ignore[reportMissingImports]
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2lab,
    label2id=lab2id
)

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    return {'accuracy': acc, 'macro_f1': f1}

strategy_key = 'eval_strategy' if hasattr(TrainingArguments, 'eval_strategy') else 'evaluation_strategy'
train_kwargs = {
    'output_dir': './colab_checkpoints',
    'num_train_epochs': 5,
    'per_device_train_batch_size': 8,
    'per_device_eval_batch_size': 8,
    'learning_rate': 3e-5,
    'weight_decay': 0.01,
    strategy_key: 'epoch',
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'macro_f1',
    'report_to': [],
    'seed': 42
}
training_args = TrainingArguments(**train_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

print('Training started on Colab runtime...')
trainer.train()


In [ ]:
# Step 7: Classification Report & Dark-Mode Confusion Matrix
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

preds = trainer.predict(test_ds)
y_pred = preds.predictions.argmax(-1)
y_true = preds.label_ids

target_names = [id2lab[i] for i in range(len(labels))]
print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(9, 8), dpi=150)
fig.patch.set_facecolor('#0b0f19')
ax.set_facecolor('#0b0f19')

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap='Blues', ax=ax, xticks_rotation=45, colorbar=True)

plt.title('Re:Learn IDE — DistilBERT Misconception Classifier\n(Held-Out Test Set with Paired Hard Negatives)', pad=15)
plt.tight_layout()
plt.savefig('confusion_matrix_distilbert.png', facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# Step 8: Export Production Model
model.save_pretrained('./relearn_distilbert_prod')
tokenizer.save_pretrained('./relearn_distilbert_prod')
print('✅ Exported production weights to ./relearn_distilbert_prod')
